# 03C — Changement de domaine et verrou spatial

Ce notebook exécute les tâches 25–26 du protocole, exclusivement à partir des
prédictions OOF de 03B sur les batches 1–2. Il attribue un statut explicite à
chacun des huit tracks, sans supprimer les domaines non soutenus, puis calibre
un post-traitement spatial global. Sur les images pures, la vérité pixel est
exacte à l’intérieur du masque segmenté : `almond*` est négatif et `peanut*`
est positif ; l’arrière-plan n’est jamais évalué.


## 1 — Gouvernance et chargement strict de 03B


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
if (current_dir / "src").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise RuntimeError("Launch 03C from the repository or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import verify_frozen_protocol
from src.utils import save_parquet
from src.workflows.projection_domain_audit import (
    build_projection_eligibility,
    build_projection_shift_diagnostics,
)
from src.workflows.spatial_postprocessing_calibration import (
    build_spatial_calibration_input,
    build_spatial_candidate_grid,
    calibrate_spatial_postprocessing,
    verify_spatial_postprocessing_lock,
)

results_tag = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
protocol_dir = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
input_dir = PROJECT_ROOT / "results" / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{results_tag}"
output_dir = PROJECT_ROOT / "results" / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{results_tag}"
output_dir.mkdir(parents=True, exist_ok=True)
input_paths = {
    key: input_dir / filename
    for key, filename in expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES.items()
}
output_paths = {
    key: output_dir / filename
    for key, filename in expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES.items()
}

verify_frozen_protocol(protocol_dir, strict=True)
protocol_lock = json.loads(
    (protocol_dir / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(encoding="utf-8")
)
protocol_hash = str(protocol_lock["lock_sha256"])

required_03b = (
    "track_contracts",
    "oof_object_predictions",
    "oof_pixel_predictions",
    "projection_shift",
    "calibration_domain",
    "calibration_audit",
)
missing_03b = [key for key in required_03b if not input_paths[key].exists()]
if missing_03b:
    raise FileNotFoundError(
        "03C requires a complete rerun of 03B; missing artifacts: "
        f"{missing_03b}"
    )

track_contracts = pd.read_parquet(input_paths["track_contracts"])
oof_objects = pd.read_parquet(input_paths["oof_object_predictions"])
oof_pixels = pd.read_parquet(input_paths["oof_pixel_predictions"])
projection_shift = pd.read_parquet(input_paths["projection_shift"])
calibration_domain = pd.read_parquet(input_paths["calibration_domain"])
calibration_audit = pd.read_parquet(input_paths["calibration_audit"])
if set(calibration_domain["protocol_hash"].astype(str)) != {protocol_hash}:
    raise RuntimeError("03B calibration_domain does not match the frozen protocol.")
selection_audit = calibration_audit.loc[
    calibration_audit["audit_type"].eq("selection_funnel")
].copy()
if set(selection_audit["track_id"].astype(str)) != {f"E{i}" for i in range(1, 9)}:
    raise RuntimeError("03B audit must report exactly E1-E8.")
unsupported_internal_tracks = {
    str(row.evaluation_track): f"internal_calibration:{row.failure_reason}"
    for row in selection_audit.loc[
        selection_audit["track_status"].eq("unsupported")
    ].itertuples(index=False)
}
calibrated_tracks = set(
    selection_audit.loc[
        selection_audit["track_status"].eq("calibrated"),
        "evaluation_track",
    ].astype(str)
)
if set(calibration_domain["evaluation_track"].astype(str)) != calibrated_tracks:
    raise RuntimeError("03B executable domain does not match its calibration audit.")

object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=True,
)


## 2 — Tâche 25 : diagnostic train–projection


In [ ]:
projection_diagnostics = build_projection_shift_diagnostics(
    oof_objects,
    oof_pixels,
    calibration_domain,
    projection_shift,
    object_db=object_db,
    protocol_hash=protocol_hash,
)
projection_eligibility = build_projection_eligibility(
    projection_diagnostics,
    calibration_domain,
    protocol_hash=protocol_hash,
    expected_tracks=track_contracts["evaluation_track"].astype(str),
    unsupported_tracks=unsupported_internal_tracks,
)
save_parquet(
    projection_diagnostics,
    output_paths["projection_shift_diagnostics"],
)
save_parquet(
    projection_eligibility,
    output_paths["projection_eligibility"],
)

if len(projection_eligibility) != 8:
    raise RuntimeError("Every one of E1-E8 must have one eligibility status.")
allowed_statuses = {
    "eligible",
    "eligible_with_warning",
    "unsupported_domain_shift",
    "unsupported_internal_calibration",
}
if not set(projection_eligibility["eligibility_status"]).issubset(allowed_statuses):
    raise RuntimeError("Unknown domain eligibility status.")
eligibility_diagnostics = projection_diagnostics.loc[
    projection_diagnostics["stratum_type"].isin(
        expcfg.PROJECTION_DOMAIN_ELIGIBILITY_DIMENSIONS
    )
]
if not eligibility_diagnostics["n_observations"].eq(
    eligibility_diagnostics["n_target"]
).all():
    raise RuntimeError(
        "Eligibility diagnostics must use target projections only."
    )
complete_diagnostics = eligibility_diagnostics.loc[
    eligibility_diagnostics["diagnostic_status"].eq("ok")
]
shift_columns = [
    "pca_centroid_shift",
    "t2_standardized_shift",
    "q_standardized_shift",
    "rule_limit_standardized_shift",
    "normalized_ratio_standardized_shift",
    "simca_margin_standardized_shift",
]
if complete_diagnostics[shift_columns].isna().any().any():
    raise RuntimeError(
        "A supported target diagnostic silently lost a shift value."
    )

display(projection_eligibility)
display(
    projection_diagnostics.sort_values(
        ["out_of_domain_rate", "target_rejection_rate"],
        ascending=False,
    ).head(expcfg.INTERNAL_CALIBRATION_MAX_ROWS_TO_DISPLAY)
)


Les lignes `unsupported_domain_shift` sont conservées dans les deux sorties.
Elles constituent un résultat scientifique explicite et ne sont jamais
supprimées silencieusement du tableau d’éligibilité. Pour les strates qui
déterminent l’éligibilité (`overall` et `fold`), la comparaison au train
SIMCA porte uniquement sur les projections cibles `peanut` : le rejet voulu
des `almond` ne peut donc pas être confondu avec un changement de domaine.
Les strates descriptives, notamment `truth_class=non_target`, conservent
l’information sur les non-cibles. Une variance petite mais non nulle est
standardisée sans seuil absolu ; une variance exactement nulle reste un
signal non borné explicite et ne devient jamais silencieusement `NaN`.


## 3 — Tâche 26 : vérité automatique des images pures

La construction des cartes est vectorisée séparément pour chaque
`domain_config_id` et chaque image. Les coordonnées dupliquées, modes de
décision inconnus, marges non finies et seuils manquants sont bloquants :
aucune affectation vectorisée ne peut écraser un pixel ou le faire changer
de configuration. La couche incertaine est conservée mais exclue des
pixels scorés ; elle n’est donc assimilée ni à une cible ni à une non-cible.
Le verrou spatial est appris uniquement sur les tracks `eligible` ou
`eligible_with_warning` : un track déjà déclaré non supporté ne peut pas
influencer le post-traitement des tracks effectivement exécutables.


In [ ]:
spatial_supported_statuses = {"eligible", "eligible_with_warning"}
spatial_supported_tracks = set(
    projection_eligibility.loc[
        projection_eligibility["eligibility_status"].isin(
            spatial_supported_statuses
        ),
        "evaluation_track",
    ].astype(str)
)
spatial_calibration_domain = calibration_domain.loc[
    calibration_domain["evaluation_track"].astype(str).isin(
        spatial_supported_tracks
    )
].copy()
if spatial_calibration_domain.loc[
    spatial_calibration_domain["projection_level"].astype(str).eq(
        "pixel_projection"
    )
].empty:
    raise RuntimeError(
        "No domain-supported pixel track remains for spatial calibration."
    )
spatial_input = build_spatial_calibration_input(
    oof_pixels,
    spatial_calibration_domain,
    image_db,
)
observed_batches = set(
    pd.to_numeric(spatial_input["batch"], errors="raise").astype(int)
)
if observed_batches.intersection(expcfg.SPATIAL_CALIBRATION_FORBIDDEN_BATCHES):
    raise RuntimeError("Batch 3 or 4 entered spatial calibration.")
if set(spatial_input["truth_level"].astype(str)) != {
    expcfg.SPATIAL_CALIBRATION_TRUTH_SOURCE
}:
    raise RuntimeError("Spatial truth is not the exact pure-image contract.")

display(
    spatial_input.groupby(
        ["evaluation_track", "batch", "truth_level"], as_index=False
    ).agg(
        n_images=("source_image", "nunique"),
        n_pixels=("row", "size"),
        uncertain_rate=("raw_uncertain", "mean"),
    )
)


## 4 — Grille spatiale, comparaison brute et verrou


In [ ]:
spatial_grid = build_spatial_candidate_grid()
display(spatial_grid)

spatial_metrics, fragment_size_classes, spatial_lock = (
    calibrate_spatial_postprocessing(
        spatial_input,
        image_db,
        protocol_hash=protocol_hash,
        candidate_grid=spatial_grid,
    )
)
save_parquet(
    spatial_metrics,
    output_paths["spatial_calibration_metrics"],
    optimize=False,
)
save_parquet(
    fragment_size_classes,
    output_paths["fragment_size_classes"],
    optimize=False,
)
output_paths["spatial_postprocessing_lock"].write_text(
    json.dumps(spatial_lock, indent=2, sort_keys=True),
    encoding="utf-8",
)
persisted_spatial_metrics = load_parquet(
    output_paths["spatial_calibration_metrics"]
)
persisted_fragment_size_classes = load_parquet(
    output_paths["fragment_size_classes"]
)
verify_spatial_postprocessing_lock(
    spatial_lock,
    persisted_spatial_metrics,
    persisted_fragment_size_classes,
)

selected_id = str(spatial_lock["selected_parameters"]["spatial_candidate_id"])
selected_connectivity = int(
    spatial_lock["selected_parameters"]["connectivity"]
)
comparison = spatial_metrics.loc[
    (
        spatial_metrics["map_variant"].eq("raw")
        & spatial_metrics["connectivity"].eq(selected_connectivity)
    )
    | spatial_metrics["spatial_candidate_id"].astype(str).eq(selected_id)
]
display(spatial_lock["selected_parameters"])
comparison_by_track = comparison.groupby(
    ["map_variant", "evaluation_track"], as_index=False
).agg(
    dice=("dice", "mean"),
    iou=("iou", "mean"),
    pixel_precision=("pixel_precision", "mean"),
    pixel_recall=("pixel_recall", "mean"),
    component_precision=("component_precision", "mean"),
    component_recall=("component_recall", "mean"),
    split_rate=("split_rate", "mean"),
    merge_rate=("merge_rate", "mean"),
    smallest_fragment_recall=("smallest_fragment_recall", "mean"),
    uncertain_pixel_rate=("uncertain_pixel_rate", "mean"),
)
display(
    comparison_by_track.groupby("map_variant", as_index=False).agg(
        dice=("dice", "mean"),
        iou=("iou", "mean"),
        pixel_precision=("pixel_precision", "mean"),
        pixel_recall=("pixel_recall", "mean"),
        component_precision=("component_precision", "mean"),
        component_recall=("component_recall", "mean"),
        split_rate=("split_rate", "mean"),
        merge_rate=("merge_rate", "mean"),
        smallest_fragment_recall=("smallest_fragment_recall", "mean"),
        uncertain_pixel_rate=("uncertain_pixel_rate", "mean"),
    )
)


## 5 — Contrôles bloquants finaux


In [ ]:
pixel_tracks = set(
    spatial_calibration_domain.loc[
        spatial_calibration_domain["projection_level"].astype(str).eq("pixel_projection"),
        "evaluation_track",
    ].astype(str)
)
locked_tracks = set(
    spatial_metrics.loc[
        spatial_metrics["is_locked_candidate"].astype(bool),
        "evaluation_track",
    ].astype(str)
)
if locked_tracks != pixel_tracks:
    raise RuntimeError(
        "Every calibrated pixel track must be represented in the spatial lock: "
        f"missing={sorted(pixel_tracks - locked_tracks)}"
    )
expected_domain_ids = set(
    spatial_calibration_domain.loc[
        spatial_calibration_domain["projection_level"].astype(str).eq(
            "pixel_projection"
        ),
        "domain_config_id",
    ].astype(str)
)
locked_domain_ids = set(
    spatial_metrics.loc[
        spatial_metrics["is_locked_candidate"].astype(bool),
        "domain_config_id",
    ].astype(str)
)
if locked_domain_ids != expected_domain_ids:
    raise RuntimeError(
        "The spatial lock does not cover every executable pixel domain."
    )
if spatial_lock.get("selection_weighting") != (
    "equal_evaluation_track_after_equal_domain_configuration"
):
    raise RuntimeError("The spatial lock is not track-balanced.")
variants = set(spatial_metrics["map_variant"].astype(str))
if variants != {"raw", "postprocessed"}:
    raise RuntimeError("Raw and postprocessed maps must both be evaluated.")
if spatial_metrics.loc[
    spatial_metrics["is_locked_candidate"].astype(bool),
    "uncertain_pixel_rate",
].isna().all():
    raise RuntimeError("The uncertainty layer was not audited.")

print(
    "03C completed: all eight domain statuses are explicit and the OOF "
    "spatial post-processing parameters are locked before 04A/04B."
)
